In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas 
import pickle
import pandas as pd
import os


In [3]:
path = "/lustre/groups/ml01/workspace/ot_perturbation/data/embeddings"
file = "cache/pathway_embeddings/stringdb.human.high.pkl"

In [1]:
import jax
jax.__version__

'0.7.0'

In [4]:
import functools
import os
import sys
import traceback
from typing import Dict, Literal, Optional, Tuple

import cellflow
from cellflow import preprocessing as cfpp
import scanpy as sc
import numpy as np
import functools
from ott.solvers import utils as solver_utils
import optax
from omegaconf import OmegaConf
from typing import NamedTuple, Any
import hydra
import wandb
import pandas as pd
import time
import anndata as ad

from numpy.typing import ArrayLike


def add_subgroup_annotations(adata_train, adata): 

    train_conditions = adata_train.obs.condition.str.replace("+ctrl", "").str.replace("ctrl+", "").unique()

    assert not adata[adata.obs.condition != "ctrl"].obs.condition.isin(train_conditions).any()

    mask_single_perturbation = adata.obs.condition.str.contains("ctrl")
    mask_double_perturbation_seen_0 = (
        ~adata.obs.condition.str.contains("ctrl") & 
        ~adata.obs.gene_1.isin(train_conditions) & 
        ~adata.obs.gene_2.isin(train_conditions)
    )
    mask_double_perturbation_seen_1 = (
        ~adata.obs.condition.str.contains("ctrl") & 
        (
            (adata.obs.gene_1.isin(train_conditions) & ~adata.obs.gene_2.isin(train_conditions)) | 
            (~adata.obs.gene_1.isin(train_conditions) & adata.obs.gene_2.isin(train_conditions))
        )
    )
    mask_double_perturbation_seen_2 = (
        ~adata.obs.condition.str.contains("ctrl") & 
        adata.obs.gene_1.isin(train_conditions) & 
        adata.obs.gene_2.isin(train_conditions)
    )
    adata.obs.loc[mask_single_perturbation, "subgroup"] = "single"
    adata.obs.loc[mask_double_perturbation_seen_0, "subgroup"] = "double_seen_0"
    adata.obs.loc[mask_double_perturbation_seen_1, "subgroup"] = "double_seen_1"
    adata.obs.loc[mask_double_perturbation_seen_2, "subgroup"] = "double_seen_2"

def add_embeddings(adata):
    with open('/lustre/groups/ml01/workspace/ot_perturbation/data/embeddings/gene_nargab.pkl', 'rb') as f:
        nargab_emb = pickle.load(f)
    adata.uns["nargab"] = nargab_emb


/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/optuna/study/_optimize.py:29: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from optuna import progress_bar as pbar_module
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWa

In [5]:
split = "0"
adata_train_path = f"/home/haicu/soeren.becker/repos/ot_pert_reproducibility/norman2019/norman_preprocessed_adata/adata_train_pca_50_split_{split}.h5ad"
adata_test_path = f"/home/haicu/soeren.becker/repos/ot_pert_reproducibility/norman2019/norman_preprocessed_adata/adata_val_pca_50_split_{split}.h5ad"
adata_ood_path = f"/home/haicu/soeren.becker/repos/ot_pert_reproducibility/norman2019/norman_preprocessed_adata/adata_test_pca_50_split_{split}.h5ad"
adata_train = sc.read_h5ad(adata_train_path)
adata_test = sc.read_h5ad(adata_test_path)
adata_ood = sc.read_h5ad(adata_ood_path)

add_embeddings(adata_train)
add_embeddings(adata_test)
add_embeddings(adata_ood)


add_subgroup_annotations(adata_train, adata_ood)
add_subgroup_annotations(adata_train, adata_ood)

adata_ood_single = adata_ood[(adata_ood.obs["kategory"] == "ctrl") | (adata_ood.obs["subgroup"]=="single")]
adata_ood_double_seen_0 = adata_ood[(adata_ood.obs["kategory"] == "ctrl") | (adata_ood.obs["subgroup"]=="double_seen_0")]
adata_ood_double_seen_1 = adata_ood[(adata_ood.obs["kategory"] == "ctrl") | (adata_ood.obs["subgroup"]=="double_seen_1")]
adata_ood_double_seen_2 = adata_ood[(adata_ood.obs["kategory"] == "ctrl") | (adata_ood.obs["subgroup"]=="double_seen_2")]


In [6]:
cf = cellflow.model.CellFlow(adata_train, solver="otfm")

In [7]:
# Prepare the training data and perturbation conditions
perturbation_covariates = {"target_gene": ["gene_1", "gene_2"]}
perturbation_covariate_reps = {"target_gene": "esm2"}

cf.prepare_data(
    sample_rep="X_pca",
    control_key="control",
    perturbation_covariates=perturbation_covariates,
    perturbation_covariate_reps=perturbation_covariate_reps,
    sample_covariates=None,
    sample_covariate_reps=None,
    split_covariates=None
)

[########################################] | 100% Completed | 234.55 ms
[########################################] | 100% Completed | 211.75 ms
[########################################] | 100% Completed | 101.70 ms


In [8]:
match_fn = functools.partial(
        solver_utils.match_linear,
    )

In [9]:
optimizer = optax.MultiSteps(optax.adam(1e-4), 20)
probability_path= {"constant_noise": 0.0}


layers_before_pool = { 
  "target_gene": {
    "layer_type": "mlp",
    "dims": [1024, 1024],
    "dropout_rate": 0.5,
  }
}
layers_after_pool = {
  "layer_type": "mlp",
  "dims": [1024, 1024],
  "dropout_rate": 0.2,
}

solver_kwargs = {"ema": 1.0}

In [10]:
cf.prepare_model(
    condition_mode="deterministic",
)

In [11]:
cf.prepare_validation_data(
    adata_ood_single,
    name="ood_single",
    n_conditions_on_log_iteration=None,
    n_conditions_on_train_end=None,
    predict_kwargs = None
)
cf.prepare_validation_data(
    adata_ood_double_seen_0,
    name="ood_doube_seen_0",
    n_conditions_on_log_iteration=None,
    n_conditions_on_train_end=None,
    predict_kwargs = None
)
cf.prepare_validation_data(
    adata_ood_double_seen_1,
    name="ood_double_seen_1",
    n_conditions_on_log_iteration=None,
    n_conditions_on_train_end=None,
    predict_kwargs = None
)

cf.prepare_validation_data(
    adata_ood_double_seen_2,
    name="ood_double_seen_2",
    n_conditions_on_log_iteration=None,
    n_conditions_on_train_end=None,
    predict_kwargs = None
)

[                                        ] | 0% Completed | 210.03 us

/ictstr01/home/icb/dominik.klein/git_repos/cell_flow_perturbation/src/cellflow/data/_datamanager.py:779: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs[self._control_key] = adata.obs[self._control_key].astype("boolean")


[########################################] | 100% Completed | 101.24 ms
[########################################] | 100% Completed | 101.57 ms
[########################################] | 100% Completed | 100.59 ms
[########################################] | 100% Completed | 101.06 ms


/ictstr01/home/icb/dominik.klein/git_repos/cell_flow_perturbation/src/cellflow/data/_datamanager.py:779: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs[self._control_key] = adata.obs[self._control_key].astype("boolean")


[########################################] | 100% Completed | 101.47 ms
[########################################] | 100% Completed | 100.77 ms
[########################################] | 100% Completed | 101.18 ms


/ictstr01/home/icb/dominik.klein/git_repos/cell_flow_perturbation/src/cellflow/data/_datamanager.py:779: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs[self._control_key] = adata.obs[self._control_key].astype("boolean")


[########################################] | 100% Completed | 101.51 ms
[########################################] | 100% Completed | 101.25 ms


/ictstr01/home/icb/dominik.klein/git_repos/cell_flow_perturbation/src/cellflow/data/_datamanager.py:779: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs[self._control_key] = adata.obs[self._control_key].astype("boolean")


[########################################] | 100% Completed | 101.13 ms
[########################################] | 100% Completed | 101.50 ms
[########################################] | 100% Completed | 100.81 ms


In [12]:
metrics_callback = cellflow.training.Metrics(metrics=["r_squared", "mmd", "e_distance"])
decoded_metrics_callback = cellflow.training.PCADecodedMetrics(ref_adata=adata_train, metrics=["r_squared", "mmd", "e_distance"])
wandb_callback = cellflow.training.WandbLogger(
    project="norman", 
    out_dir="/lustre/groups/ml01/workspace/ot_perturbation/logging", 
    config={},
)
callbacks = [metrics_callback, decoded_metrics_callback, wandb_callback]

cf.train(
    num_iterations=1000,
    batch_size=1024,
    callbacks=callbacks,
    valid_freq=100,
)

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: mucdk (modality_translation) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


100%|██████████| 1000/1000 [17:05<00:00,  1.03s/it, loss=1.62] 


ood_single_e_distance_mean,▇▂█▄▂▆▁▄▂▃
ood_single_mmd_mean,█▆▆▃▂▃▁▂▁▁
ood_single_r_squared_mean,▁▆▁▅█▅█▅▆▇
pca_decoded_ood_single_e_distance_mean,▇▂█▄▂▆▁▄▂▃
pca_decoded_ood_single_mmd_mean,█▆▆▃▂▃▁▂▁▁
pca_decoded_ood_single_r_squared_mean,▂▇▁▅▇▃█▅▇▆
ood_single_e_distance_mean,45.27016
ood_single_mmd_mean,0.04398
ood_single_r_squared_mean,0.03742
pca_decoded_ood_single_e_distance_mean,45.27027
pca_decoded_ood_single_mmd_mean,0.04396


In [13]:
cf.trainer.predict_kwargs = {"max_steps": 3}


In [14]:
cf.trainer.predict_kwargs

{'max_steps': 3}

In [23]:
# Prepare the model
cf.prepare_model(
    condition_mode="deterministic",
    
    regularization=config_dict["model"]["regularization"],
    pooling=config_dict["model"]["pooling"],
    layers_before_pool=layers_before_pool,
    layers_after_pool=layers_after_pool,
    condition_embedding_dim=config_dict["model"]["condition_embedding_dim"],
    cond_output_dropout=config_dict["model"]["cond_output_dropout"],
    time_freqs=config_dict["model"]["time_freqs"],
    time_max_period=config_dict["model"]["time_max_period"],
    time_encoder_dims=config_dict["model"]["time_encoder_dims"],
    time_encoder_dropout=config_dict["model"]["time_encoder_dropout"],
    hidden_dims=config_dict["model"]["hidden_dims"],
    hidden_dropout=config_dict["model"]["hidden_dropout"],
    conditioning=config_dict["model"]["conditioning"],
    decoder_dims=config_dict["model"]["decoder_dims"],
    decoder_dropout=config_dict["model"]["decoder_dropout"],
    probability_path=probability_path,
    solver_kwargs=solver_kwargs,
    match_fn=match_fn,
    optimizer=optimizer,
    layer_norm_before_concatenation=config_dict["model"]["layer_norm_before_concatenation"],
    linear_projection_before_concatenation=config_dict["model"]["linear_projection_before_concatenation"],
)

NameError: name 'config_dict' is not defined

In [ ]:

    
    
    
    
    
    
    
    

    
    
    
    metrics_callback = cfp.training.Metrics(metrics=["r_squared", "mmd", "e_distance"])
    decoded_metrics_callback = cfp.training.PCADecodedMetrics(ref_adata=adata_train, metrics=["r_squared", "mmd", "e_distance"])
    wandb_callback = cfp.training.WandbLogger(
        project="norman", 
        out_dir="/lustre/groups/ml01/workspace/ot_perturbation/logging", 
        config=config_dict,
    )
    callbacks = [metrics_callback, decoded_metrics_callback, wandb_callback]
    
    cf.train(
        num_iterations=config_dict["training"]["num_iterations"],
        batch_size=config_dict["training"]["batch_size"],
        callbacks=callbacks,
        valid_freq=config_dict["training"]["valid_freq"],
    )